# Soft-sector reference parameters, posterior corner plots and MAP

Inspect the completed [JETSCAPE Bayesian study](https://arxiv.org/abs/2011.01430), its released reference/model inputs, and a local full-data Grad posterior rerun. Run `python scripts/fetch_soft_inputs.py` from the repository root first. Public inputs are pinned by [upstream_manifest.json](upstream_manifest.json); the posterior archive is included. This notebook reads saved results and plots the MAP comparison.


In [ ]:
from pathlib import Path
import os, sys, json
import numpy as np
import matplotlib.pyplot as plt
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "scripts/inference_tools.py").is_file())
sys.path.insert(0, str(ROOT / "scripts"))
from inference_tools import load_samples, summarize, corner_plot, approximate_joint_mode, hard_point_to_xml, sha256
OUT = ROOT / "outputs" / "inference"
OUT.mkdir(parents=True, exist_ok=True)

from soft_inputs import read_design_archive, read_model, viscosity_curves

In [ ]:
DATA = ROOT / "soft"
manifest = json.loads((DATA / "upstream_manifest.json").read_text())
cache = Path(os.environ.get("SOFT_CACHE", ROOT / ".cache/soft-sector"))
for item in manifest["files"]:
    path = cache / item["path"]
    if not path.is_file(): raise FileNotFoundError("Fetch upstream inputs with python scripts/fetch_soft_inputs.py")
    assert sha256(path) == item["sha256"]
reference = json.loads((DATA / "reference_points.json").read_text())
schema = json.loads((DATA / "observable_schema.json").read_text())
ids, names, design, ranges = read_design_archive(cache / manifest["files"][0]["path"])
assert design.shape == (500, 17)
print("Pinned upstream commit:", manifest["commit"])
print("Design shape:", design.shape, "Prior normalization range:", ranges["norm"])
print(json.dumps(reference["values"]["Pb-Pb-2760"], indent=2))

The reference tables contain 17 coordinates per collision system. Joint two-system posteriors have two normalizations and may also include a discrepancy parameter $\sigma_M$. Keep the complete column declaration when reading a chain. The coordinate `dmin3` is $d_{\min}^3$ in fm$^3$, despite the older design-label file calling it a length.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
T = np.linspace(.13, .35, 180)
for model, point in reference["values"]["Pb-Pb-2760"].items():
    eta, zeta = viscosity_curves(T, point)
    axes[0].plot(T, eta, label=model); axes[1].plot(T, zeta, label=model)
for ax, label in zip(axes, [r"$\eta/s$", r"$\zeta/s$"]):
    ax.set_xlabel("Temperature [GeV]"); ax.set_ylabel(label); ax.legend()
fig.suptitle("Upstream code-side soft reference tables, Pb–Pb 2.76 TeV")
fig.tight_layout(); fig.savefig(OUT / "soft_reference_viscosities.png", dpi=140)
plt.show()

In [ ]:
main = read_model(cache / manifest["files"][1]["path"], schema, "Pb-Pb-2760")
map_model = read_model(cache / manifest["files"][2]["path"], schema, "Pb-Pb-2760", map_record=True)
assert len(main) == len(ids)
observable = "dNch_deta"
selection = schema["training_selection"]
keep = ~np.isin(ids, selection["excluded_design_ids"])
values = main[observable]["mean"][:, 0]
finite = np.isfinite(values).all(axis=1)
print("Explicit upstream Grad exclusions:", int((~keep).sum()), "additional nonfinite rows:", int((keep & ~finite).sum()))
values = values[keep & finite]
bins = np.asarray(schema["systems"]["Pb-Pb-2760"][observable]); centers = bins.mean(axis=1)
lo, med, hi = np.quantile(values, [.05, .5, .95], axis=0)
y = map_model[observable]["mean"][0, 0]; err = map_model[observable]["err"][0, 0]
assert np.isfinite(y).all() and np.isfinite(err).all() and (err >= 0).all()
fig, ax = plt.subplots(figsize=(7, 4))
ax.fill_between(centers, lo, hi, alpha=.25, label="5–95% design envelope (not posterior)")
ax.plot(centers, med, label="Design median")
ax.errorbar(centers, y, yerr=err, marker="o", label="Stored Grad MAP simulation ± MC error")
ax.set_xlabel("Centrality [%]"); ax.set_ylabel(r"$dN_{ch}/d\eta$"); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(OUT / "soft_design_and_map.png", dpi=140)
plt.show()

This compares stored simulations with the design envelope; it is not an emulator prediction or an experimental-data fit. The Pb MAP file contains one full Pb+Au compound record with a zero Au block. The reader validates that layout explicitly instead of ignoring trailing bytes. Serialized upstream emulators and their training code are linked in the README; their legacy environment is separate from this lightweight reader environment.

In [ ]:
from inference_tools import highest_posterior_sample


## Posterior corner plots and MAP parameters

The included [posterior archive](posterior.npz) is a local full-data Grad rerun using the completed study's inputs. It has 18 coordinates, two collision-system normalizations and no `sigma_M`. Its [metadata](posterior_metadata.json) records the weights, coordinates, likelihood size and source identity. Use all rows with their importance weights.

Compare the joint MAP estimate with the values printed in [Table II of the paper](https://arxiv.org/html/2011.01430v2#S7.T2). The local reconstruction has a code-defined 146-bin likelihood and three prior-range differences listed in the README, so this is a comparison with the published result.

**Reference simulations should use the published MAP values.** The local estimate depends on practical likelihood, prior, emulator and optimizer choices; it illustrates the method and does not replace the published tune.


In [ ]:
from soft_posterior import load_posterior, corner_plot as soft_corner_plot, weighted_quantile
posterior = load_posterior(DATA / "posterior.npz")
metadata = json.loads((DATA / "posterior_metadata.json").read_text())
assert sha256(DATA / "posterior.npz") == metadata["archive_sha256"]
names = posterior["columns"]
weights = posterior["weights"]
print(metadata["case"])
print("Samples:", posterior["samples"].shape, "importance-weight ESS:", 1 / np.sum(weights**2))


In [ ]:
published = json.loads((DATA / "published_map.json").read_text())
optimized = json.loads((DATA / "map_result.json").read_text())
assert optimized["posterior_sha256"] == metadata["archive_sha256"]
assert optimized["converged"]
local_map = optimized["point"]
paper_map = published["values"]["Grad"]
scores = posterior["log_likelihood"] + posterior["log_prior"]
best_retained = highest_posterior_sample(posterior["samples"], scores, names)
print("MAP method:", optimized["method"])
print("Posterior-score improvement over best retained sample:", optimized["improvement_over_best_saved_sample"])
print("Published reference:", published["citation"])


In [ ]:
import csv
rows = []
for i, name in enumerate(names):
    q16, median, q84 = weighted_quantile(posterior["samples"][:, i], weights, [.16, .5, .84])
    rows.append({"parameter": name, "published_Grad_MAP": paper_map[name],
                 "local_MAP": local_map[name], "difference": local_map[name]-paper_map[name],
                 "posterior_median": median, "q16": q16, "q84": q84})
with (OUT / "soft_map_comparison.csv").open("w", newline="") as stream:
    writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
    writer.writeheader(); writer.writerows(rows)
print(f"{'Parameter':40s} {'Published':>10s} {'Local MAP':>10s} {'Difference':>11s}")
for row in rows:
    print(f"{row['parameter']:40s} {row['published_Grad_MAP']:10.6g} {row['local_MAP']:10.6g} {row['difference']:11.6g}")


In [ ]:
labels = [r"$N_{\rm LHC}$", r"$N_{\rm RHIC}$", r"$p$", r"$\sigma_k$", r"$w$ [fm]",
          r"$d_{\min}^3$ [fm$^3$]", r"$\tau_R$ [fm/$c$]", r"$\alpha$", r"$T_\eta$ [GeV]",
          r"$a_{\rm low}$ [GeV$^{-1}$]", r"$a_{\rm high}$ [GeV$^{-1}$]", r"$(\eta/s)_k$",
          r"$(\zeta/s)_{\max}$", r"$T_\zeta$ [GeV]", r"$w_\zeta$ [GeV]", r"$\lambda_\zeta$",
          r"$b_\pi$", r"$T_{\rm sw}$ [GeV]"]
fig = soft_corner_plot(posterior, list(range(18)), labels, local_map, paper_map)
fig.savefig(OUT / "soft_corner_full.png", dpi=150)
plt.close(fig)
selected = list(range(6))
fig = soft_corner_plot(posterior, selected, [labels[i] for i in selected], local_map, paper_map)
fig.savefig(OUT / "soft_corner_selected.png", dpi=150)
plt.show()


The blue contours contain 68% and 95% of the smoothed two-dimensional weighted histogram mass. Marginals retain the importance weights. Solid red lines show the local bounded joint MAP estimate; dashed dark lines show the paper's Grad MAP. The full 18-parameter corner is saved alongside the selected-parameter plot.

The saved [MAP result](map_result.json) maximizes the original emulator likelihood within the declared local prior box, starting from the best retained sample, the published Grad point and the weighted posterior mean. It includes parameter-dependent model covariance. It is the best local optimum among those starts; a global optimum is not established. A best retained sample, a marginal median and a KDE mode are different estimators.

To recompute the optimization, use the Python environment that loads your matching SIMS emulators:

```bash
python scripts/find_soft_map.py --sims-source /path/to/matching/js-sims-bayes \
  --posterior soft/posterior.npz --published soft/published_map.json \
  --output outputs/inference/soft_map_recomputed.json
```

The optimizer checks the dimensions, prior bounds and stored likelihood scores before running. Plotting and reading the saved comparison do not require loading serialized emulators.

**Reference simulations should use the published MAP values.** The local estimate depends on practical likelihood, prior, emulator and optimizer choices; it illustrates the method and does not replace the published tune.
